# 2. 拓扑、连接数组合与连接键（Windows 版）

语言：[English](../en/02_topologies_connectivities_and_linkages.ipynb) | **中文**

本原生 Windows 版 Notebook 构建三个经过筛选的示例，足以展示主要自由度，同时避免变成冗长的案例目录。所有可执行单元格都是 CLI PowerShell 命令块，不需要 WSL；每次构建下方都提供了注释掉的对应 Python API 调用。

## 教程系列

1. [环境与首次构建](01_environment_and_first_build.ipynb)
2. **拓扑、连接数组合与连接键示例**（本 Notebook）
3. [使用 Zeo++ 进行孔隙分析](03_zeopp_pore_analysis.ipynb)

请先运行 Windows 版 [Notebook 1](01_environment_and_first_build.ipynb)。本 Notebook 将 `Python (cofkit)` 声明为默认内核，同时也显式使用 `conda run -n cofkit`。

## 三个案例

| 案例 | 反应基团数量 | 拓扑 | 维度 | 连接键 | 主要比较点 |
|---|---:|---|---|---|---|
| TAPB + 对苯二甲醛 | 3+2 | `hcb` | 2D | 亚胺 | 节点–连接体构建 |
| TAPM + 对苯二甲醛 | 4+2 | `dia` | 3D | 亚胺 | 更高连接数的三维网络 |
| Tp + Pa-1 | 3+2 | `hcb` | 2D | β-酮烯胺 | 改变连接键化学 |

结合 Notebook 1 中的 3+3 亚胺/`hcb` 案例，本系列涵盖三种连接数组合、两种拓扑、两种维度和两类连接键。示例明确指定基元类型和拓扑 ID，使每条命令都只比较一个受控变量。

## 验证共享环境

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
& conda run -n cofkit cofkit --version
if ($LASTEXITCODE -ne 0) { throw "cofkit version check failed with exit code $LASTEXITCODE." }
& conda run -n cofkit cofkit build list-templates
if ($LASTEXITCODE -ne 0) { throw "Template listing failed with exit code $LASTEXITCODE." }

In [ ]:
# Python 等价实现（已注释）：验证共享环境。
# import cofkit
# print(cofkit.__version__)

## 案例 1——`hcb` 上的 3+2 亚胺结构

三角形 TAPB 节点通过线性对苯二甲醛连接体相连。该案例保留相同的亚胺化学和二维 `hcb` 网络，用于对比 Notebook 1 中的 3+3 节点–节点构建。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cofkitArguments = @(
  "build", "single-pair",
  "--template-id", "imine_bridge",
  "--first-smiles", "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
  "--second-smiles", "O=Cc1ccc(C=O)cc1",
  "--first-id", "tapb",
  "--second-id", "terephthaldehyde",
  "--first-motif-kind", "amine",
  "--second-motif-kind", "aldehyde",
  "--topology", "hcb",
  "--target-dimensionality", "2D",
  "--output-dir", "out/tutorial_02_builds/01_hcb_3plus2_imine"
)
& conda run -n cofkit cofkit @cofkitArguments
if ($LASTEXITCODE -ne 0) { throw "Case 1 build failed with exit code $LASTEXITCODE." }

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb", "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine", num_conformers=4,
# )
# terephthaldehyde = build_rdkit_monomer(
#     "terephthaldehyde", "terephthaldehyde",
#     "O=Cc1ccc(C=O)cc1", "aldehyde", num_conformers=4,
# )
# generator = BatchStructureGenerator(BatchGenerationConfig(
#     allowed_reactions=("imine_bridge",),
#     target_dimensionality="2D", topology_ids=("hcb",),
#     rdkit_num_conformers=4, max_workers=1, write_cif=True,
# ))
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb, terephthaldehyde, pair_id="tapb__terephthaldehyde",
#     out_dir=Path("out/tutorial_02_builds_python/01_hcb_3plus2_imine/cifs"),
# )
# print(attempted, [(item.status, item.cif_path) for item in summaries])

## 案例 2——`dia` 上的 4+2 亚胺结构

四面体 TAPM 节点将节点连接数从三提高到四。`dia` 是三维类金刚石网络，因此这里把维度显式设为 `3D`。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cofkitArguments = @(
  "build", "single-pair",
  "--template-id", "imine_bridge",
  "--first-smiles", "Nc1ccc(C(c2ccc(N)cc2)(c2ccc(N)cc2)c2ccc(N)cc2)cc1",
  "--second-smiles", "O=Cc1ccc(C=O)cc1",
  "--first-id", "tapm",
  "--second-id", "terephthaldehyde",
  "--first-motif-kind", "amine",
  "--second-motif-kind", "aldehyde",
  "--topology", "dia",
  "--target-dimensionality", "3D",
  "--output-dir", "out/tutorial_02_builds/02_dia_4plus2_imine"
)
& conda run -n cofkit cofkit @cofkitArguments
if ($LASTEXITCODE -ne 0) { throw "Case 2 build failed with exit code $LASTEXITCODE." }

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapm = build_rdkit_monomer(
#     "tapm", "TAPM",
#     "Nc1ccc(C(c2ccc(N)cc2)(c2ccc(N)cc2)c2ccc(N)cc2)cc1",
#     "amine", num_conformers=4,
# )
# terephthaldehyde = build_rdkit_monomer(
#     "terephthaldehyde", "terephthaldehyde",
#     "O=Cc1ccc(C=O)cc1", "aldehyde", num_conformers=4,
# )
# generator = BatchStructureGenerator(BatchGenerationConfig(
#     allowed_reactions=("imine_bridge",),
#     target_dimensionality="3D", topology_ids=("dia",),
#     rdkit_num_conformers=4, max_workers=1, write_cif=True,
# ))
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapm, terephthaldehyde, pair_id="tapm__terephthaldehyde",
#     out_dir=Path("out/tutorial_02_builds_python/02_dia_4plus2_imine/cifs"),
# )
# print(attempted, [(item.status, item.cif_path) for item in summaries])

## 案例 3——`hcb` 上的 3+2 β-酮烯胺结构

该案例回到二维 3+2 `hcb` 构建，但改变了连接键化学。Tp 提供三个 `keto_aldehyde` 基元，Pa-1 提供两个胺基。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$cofkitArguments = @(
  "build", "single-pair",
  "--template-id", "keto_enamine_bridge",
  "--first-smiles", "O=Cc1c(O)c(C=O)c(O)c(C=O)c1O",
  "--second-smiles", "Nc1ccc(N)cc1",
  "--first-id", "tp",
  "--second-id", "pa1",
  "--first-motif-kind", "keto_aldehyde",
  "--second-motif-kind", "amine",
  "--topology", "hcb",
  "--target-dimensionality", "2D",
  "--output-dir", "out/tutorial_02_builds/03_hcb_3plus2_keto_enamine"
)
& conda run -n cofkit cofkit @cofkitArguments
if ($LASTEXITCODE -ne 0) { throw "Case 3 build failed with exit code $LASTEXITCODE." }

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tp = build_rdkit_monomer(
#     "tp", "Tp", "O=Cc1c(O)c(C=O)c(O)c(C=O)c1O",
#     "keto_aldehyde", num_conformers=4,
# )
# pa1 = build_rdkit_monomer(
#     "pa1", "Pa-1", "Nc1ccc(N)cc1",
#     "amine", num_conformers=4,
# )
# generator = BatchStructureGenerator(BatchGenerationConfig(
#     allowed_reactions=("keto_enamine_bridge",),
#     target_dimensionality="2D", topology_ids=("hcb",),
#     rdkit_num_conformers=4, max_workers=1, write_cif=True,
# ))
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tp, pa1, pair_id="tp__pa1",
#     out_dir=Path("out/tutorial_02_builds_python/03_hcb_3plus2_keto_enamine/cifs"),
# )
# print(attempted, [(item.status, item.cif_path) for item in summaries])

## 比较生成结果

下面的简短报告会显示每个 `summary.json` 中的关键计数，并列出所有导出的 CIF，而不区分其验证类别。

In [ ]:
%%script powershell -NoProfile
$ErrorActionPreference = "Stop"
$root = (& git rev-parse --show-toplevel).Trim()
if ($LASTEXITCODE -ne 0) { throw "git rev-parse failed with exit code $LASTEXITCODE." }
Set-Location -LiteralPath $root
$summaryFiles = Get-ChildItem -Path "out/tutorial_02_builds/*/summary.json" -File | Sort-Object FullName
foreach ($summary in $summaryFiles) {
  Write-Host ""
  Write-Host $summary.FullName
  Select-String -LiteralPath $summary.FullName -Pattern '"(template_id|target_dimensionality|attempted_structures|successful_structures|cifs_written)"' | ForEach-Object { $_.Line }
}
Write-Host "--- exported CIF files ---"
Get-ChildItem -LiteralPath "out/tutorial_02_builds" -Recurse -File -Filter "*.cif" | Sort-Object FullName | ForEach-Object { $_.FullName }

In [ ]:
# Python 等价实现（已注释）：以结构化数据形式完成相同比较。
# import json
# from pathlib import Path
# root = Path("out/tutorial_02_builds")
# for summary_path in sorted(root.glob("*/summary.json")):
#     report = json.loads(summary_path.read_text())
#     print(summary_path.parent.name, {
#         key: report[key]
#         for key in ("template_id", "target_dimensionality",
#                     "attempted_structures", "successful_structures", "cifs_written")
#     })
# print(*sorted(root.rglob("*.cif")), sep="\n")

## 解读比较结果

- 连接数决定哪些拓扑构建器在化学上兼容；显式指定拓扑也不能覆盖不匹配的基元数量。
- `target_dimensionality` 应与 `topology` 一致。`dia` 案例明确为三维，而 `hcb` 案例为二维。
- 连接键模板同时控制角色匹配和原子级成键实现。只改变 `--template-id` 并不够，单体还必须具有该模板所需的基元类型。
- 生成的 CIF 是起始结构。请阅读各摘要中的验证标记，并在下游模拟前按需优化和重新验证。

继续学习 [Notebook 3](03_zeopp_pore_analysis.ipynb)。它会有意分析 Notebook 1 的 TAPB–TFB CIF，从而保证输入可预测。